# Stock Data Refresh Notebook

This notebook automatically refreshes stock data with smart update logic.

## Features
- **Smart Update Logic**: Only updates if data is older than last week
- **Multi-Stock Support**: Updates all configured stocks
- **Data Validation**: Checks data quality and completeness
- **Progress Tracking**: Shows update status for each stock
- **Error Handling**: Continues processing even if some stocks fail


## 1. Setup and Configuration


In [1]:
import sys
import os
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Import our modules
from src.data_preprocess.stock_data_loader import StockDataLoader

print("✅ Setup complete!")


✅ Setup complete!


In [2]:
# Configuration
REFRESH_THRESHOLD_DAYS = 7  # Update if data is older than 7 days
FORCE_REFRESH = False  # Set to True to force update regardless of age

print(f"🔄 Data Refresh Configuration:")
print(f"   • Refresh threshold: {REFRESH_THRESHOLD_DAYS} days")
print(f"   • Force refresh: {FORCE_REFRESH}")
print(f"   • Current date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


🔄 Data Refresh Configuration:
   • Refresh threshold: 7 days
   • Force refresh: False
   • Current date: 2025-10-04 22:43:52


## 2. Check Current Data Status


In [3]:
# Initialize data loader
data_loader = StockDataLoader()

# Load configuration to get stock symbols
config = data_loader.config
stock_symbols = config.get('stocks', {}).get('symbols', [])

print(f"📊 Stock Configuration:")
print(f"   • Total stocks configured: {len(stock_symbols)}")
print(f"   • Stock symbols: {', '.join(stock_symbols[:10])}{'...' if len(stock_symbols) > 10 else ''}")

# Check current data status
print(f"\n🔍 Checking current data status...")

data_status = {}
needs_update = []

for symbol in stock_symbols:
    try:
        # Load existing data
        stock_data = data_loader.load_saved_data([symbol])
        
        if symbol in stock_data and not stock_data[symbol].empty:
            data = stock_data[symbol]
            
            # Get the latest date
            if 'Date' in data.columns:
                data['Date'] = pd.to_datetime(data['Date'], utc=True).dt.tz_localize(None)
                latest_date = data['Date'].max()
            else:
                latest_date = data.index.max() if isinstance(data.index, pd.DatetimeIndex) else None
            
            if latest_date:
                days_old = (datetime.now() - latest_date).days
                data_status[symbol] = {
                    'latest_date': latest_date,
                    'days_old': days_old,
                    'needs_update': days_old > REFRESH_THRESHOLD_DAYS or FORCE_REFRESH,
                    'rows': len(data)
                }
                
                if data_status[symbol]['needs_update']:
                    needs_update.append(symbol)
            else:
                data_status[symbol] = {
                    'latest_date': None,
                    'days_old': None,
                    'needs_update': True,
                    'rows': len(data)
                }
                needs_update.append(symbol)
        else:
            data_status[symbol] = {
                'latest_date': None,
                'days_old': None,
                'needs_update': True,
                'rows': 0
            }
            needs_update.append(symbol)
            
    except Exception as e:
        print(f"   ❌ {symbol}: Error checking data - {e}")
        data_status[symbol] = {
            'latest_date': None,
            'days_old': None,
            'needs_update': True,
            'rows': 0,
            'error': str(e)
        }
        needs_update.append(symbol)

print(f"\n📋 Data Status Summary:")
print(f"   • Total stocks: {len(stock_symbols)}")
print(f"   • Need update: {len(needs_update)}")
print(f"   • Up to date: {len(stock_symbols) - len(needs_update)}")


INFO:src.data_preprocess.stock_data_loader:Configuration loaded from /Users/stephenzhang/Documents/code/stock-forecast/src/data_preprocess/../../config/stocks_config.yaml


📊 Stock Configuration:
   • Total stocks configured: 0
   • Stock symbols: 

🔍 Checking current data status...

📋 Data Status Summary:
   • Total stocks: 0
   • Need update: 0
   • Up to date: 0


## 3. Display Detailed Status


In [4]:
# Display detailed status for all stocks
print("📊 Detailed Data Status:")
print(f"{'Symbol':<8} {'Latest Date':<12} {'Days Old':<10} {'Rows':<8} {'Status':<12}")
print("-" * 60)

for symbol in stock_symbols:
    status = data_status[symbol]
    
    if status['latest_date']:
        latest_str = status['latest_date'].strftime('%Y-%m-%d')
        days_str = f"{status['days_old']} days"
        rows_str = f"{status['rows']:,}"
    else:
        latest_str = "No data"
        days_str = "N/A"
        rows_str = "0"
    
    if status['needs_update']:
        status_str = "🔄 Needs Update"
    else:
        status_str = "✅ Up to Date"
    
    print(f"{symbol:<8} {latest_str:<12} {days_str:<10} {rows_str:<8} {status_str:<12}")

print(f"\n📈 Summary:")
if needs_update:
    print(f"   🔄 Stocks needing update: {', '.join(needs_update)}")
else:
    print(f"   ✅ All stocks are up to date!")


📊 Detailed Data Status:
Symbol   Latest Date  Days Old   Rows     Status      
------------------------------------------------------------

📈 Summary:
   ✅ All stocks are up to date!


## 4. Smart Data Update


In [5]:
# Perform smart updates only for stocks that need it
if needs_update:
    print(f"🔄 Starting smart data update for {len(needs_update)} stocks...")
    
    update_results = {}
    successful_updates = 0
    failed_updates = 0
    
    for i, symbol in enumerate(needs_update, 1):
        print(f"\n📈 [{i}/{len(needs_update)}] Updating {symbol}...")
        
        try:
            # Check if we have existing data to determine update strategy
            existing_data = None
            if symbol in data_status and data_status[symbol]['rows'] > 0:
                try:
                    existing_data = data_loader.load_saved_data([symbol])
                    if symbol in existing_data:
                        existing_data = existing_data[symbol]
                except:
                    existing_data = None
            
            if existing_data is not None and not existing_data.empty:
                # Incremental update
                print(f"   📊 Incremental update (existing data: {len(existing_data)} rows)")
                result = data_loader.update_stock_data(symbol)
            else:
                # Full download
                print(f"   📥 Full download (no existing data)")
                result = data_loader.download_stock_data([symbol])
            
            # Verify the update
            updated_data = data_loader.load_saved_data([symbol])
            if symbol in updated_data and not updated_data[symbol].empty:
                new_rows = len(updated_data[symbol])
                old_rows = data_status[symbol]['rows']
                added_rows = new_rows - old_rows
                
                update_results[symbol] = {
                    'status': 'success',
                    'old_rows': old_rows,
                    'new_rows': new_rows,
                    'added_rows': added_rows,
                    'latest_date': updated_data[symbol]['Date'].max() if 'Date' in updated_data[symbol].columns else None
                }
                
                print(f"   ✅ Success: {old_rows} → {new_rows} rows (+{added_rows})")
                successful_updates += 1
            else:
                raise Exception("No data after update")
                
        except Exception as e:
            print(f"   ❌ Failed: {e}")
            update_results[symbol] = {
                'status': 'failed',
                'error': str(e)
            }
            failed_updates += 1
    
    print(f"\n📊 Update Summary:")
    print(f"   ✅ Successful updates: {successful_updates}")
    print(f"   ❌ Failed updates: {failed_updates}")
    print(f"   📈 Total processed: {len(needs_update)}")
    
else:
    print("✅ No updates needed - all data is current!")
    update_results = {}


✅ No updates needed - all data is current!


## 5. Final Status Check


In [6]:
# Final status check after updates
print("🔍 Final data status check...")

final_status = {}
for symbol in stock_symbols:
    try:
        stock_data = data_loader.load_saved_data([symbol])
        
        if symbol in stock_data and not stock_data[symbol].empty:
            data = stock_data[symbol]
            
            # Get the latest date
            if 'Date' in data.columns:
                data['Date'] = pd.to_datetime(data['Date'], utc=True).dt.tz_localize(None)
                latest_date = data['Date'].max()
            else:
                latest_date = data.index.max() if isinstance(data.index, pd.DatetimeIndex) else None
            
            days_old = (datetime.now() - latest_date).days if latest_date else None
            
            final_status[symbol] = {
                'latest_date': latest_date,
                'days_old': days_old,
                'rows': len(data),
                'is_current': days_old is not None and days_old <= REFRESH_THRESHOLD_DAYS
            }
        else:
            final_status[symbol] = {
                'latest_date': None,
                'days_old': None,
                'rows': 0,
                'is_current': False
            }
            
    except Exception as e:
        final_status[symbol] = {
            'latest_date': None,
            'days_old': None,
            'rows': 0,
            'is_current': False,
            'error': str(e)
        }

# Display final status
print(f"\n📊 Final Data Status:")
print(f"{'Symbol':<8} {'Latest Date':<12} {'Days Old':<10} {'Rows':<8} {'Status':<12}")
print("-" * 60)

current_count = 0
for symbol in stock_symbols:
    status = final_status[symbol]
    
    if status['latest_date']:
        latest_str = status['latest_date'].strftime('%Y-%m-%d')
        days_str = f"{status['days_old']} days"
        rows_str = f"{status['rows']:,}"
    else:
        latest_str = "No data"
        days_str = "N/A"
        rows_str = "0"
    
    if status['is_current']:
        status_str = "✅ Current"
        current_count += 1
    else:
        status_str = "⚠️ Outdated"
    
    print(f"{symbol:<8} {latest_str:<12} {days_str:<10} {rows_str:<8} {status_str:<12}")

print(f"\n📈 Final Summary:")
print(f"   ✅ Current stocks: {current_count}/{len(stock_symbols)}")
print(f"   ⚠️ Outdated stocks: {len(stock_symbols) - current_count}/{len(stock_symbols)}")
print(f"   📅 Refresh threshold: {REFRESH_THRESHOLD_DAYS} days")

if current_count == len(stock_symbols):
    print(f"\n🎉 All stocks are up to date!")
else:
    print(f"\n⚠️ Some stocks still need attention.")


🔍 Final data status check...

📊 Final Data Status:
Symbol   Latest Date  Days Old   Rows     Status      
------------------------------------------------------------

📈 Final Summary:
   ✅ Current stocks: 0/0
   ⚠️ Outdated stocks: 0/0
   📅 Refresh threshold: 7 days

🎉 All stocks are up to date!


## 6. Usage Instructions

### **How to Use This Notebook:**

1. **Regular Updates**: Run all cells to check and update data as needed
2. **Force Refresh**: Set `FORCE_REFRESH = True` to update all stocks regardless of age
3. **Custom Threshold**: Change `REFRESH_THRESHOLD_DAYS` to adjust update frequency

### **Smart Update Logic:**

- **✅ Skip Update**: If data is newer than threshold (default: 7 days)
- **🔄 Incremental Update**: If existing data found, only download new data
- **📥 Full Download**: If no existing data, download complete history
- **❌ Error Handling**: Continue processing other stocks if one fails

### **Configuration Options:**

```python
REFRESH_THRESHOLD_DAYS = 7    # Update if data is older than 7 days
FORCE_REFRESH = False         # Set to True to force all updates
```

### **Automation:**

This notebook can be scheduled to run automatically:
- **Daily**: Check for updates every day
- **Weekly**: Run once per week to refresh data
- **Custom**: Set your preferred schedule

### **Benefits:**

- ⚡ **Efficient**: Only updates when necessary
- 🔄 **Incremental**: Preserves existing data, adds only new data
- 📊 **Comprehensive**: Handles all configured stocks
- 🛡️ **Robust**: Continues processing even if some stocks fail
- 📈 **Transparent**: Clear status reporting and progress tracking
